# LLM Rerankers: Listwise Permutation Objectives and RankGPT

The narrative companion to `llm_listwise_rerankers.py`, which **owns every number** — this notebook imports
it and walks the topic section by section. The predecessor (`lambdarank-lambdamart-listwise`) scored each
document from a *fixed* feature vector, blind to the other documents except through the loss. The LLM
reranker drops that: feed the whole candidate list into one model and it emits a **permutation** directly,
scoring documents in each other's context.

**The load-bearing constraint:** this site bakes only reproducible numbers and never calls a cloud service.
There is **no real LLM call** — the reranker is a *seeded noisy permutation oracle* that corrupts the ideal
ranking with Plackett–Luce noise (temperature $\tau$) and a position-dependent bias. Every provable claim is
therefore **algorithmic** (sorting guarantees, aggregation variance reduction, distillation cost), never
"the LLM is most accurate" (empirical, out of scope).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np
import llm_listwise_rerankers as M

corpus = M._corpus()
print("docs", corpus["n_docs"], "queries", corpus["n_queries"], "TOPK", M.TOPK)
print("pool n", M.POOL, "window w", M.WINDOW, "step s", M.STEP, "passes", M.N_PASSES)
print("train/test split", len(corpus["train_q"]), "/", len(corpus["test_q"]))

## 1 — The listwise permutation objective (the ListMLE bridge)

We model the LLM as a **Plackett–Luce sampler** at temperature $\tau$: sorting by Gumbel-perturbed
log-abilities is a PL draw. The abilities are $a_i = -\text{rank}_i$ in the ideal order $\pi^\*$, so as
$\tau\to 0$ the deterministic abilities dominate and the sample reproduces $\pi^\*$ **byte-for-byte** (the
imported `optimal_permutation`); as $\tau\to\infty$ the Gumbel noise dominates and the order is uniform.
The negative log-likelihood of an emitted permutation is exactly the imported ListMLE loss.

In [ ]:
q = corpus["train_q"][0]
docs = M.candidate_set(corpus, q)
star = M.optimal_permutation(corpus, q, docs)
emitted0 = M.noisy_oracle_perm(corpus, q, docs, M.TAU_COLLAPSE, np.random.default_rng([M.SEED, 0]))
print("tau->0 sample == pi* :", emitted0 == star)

rng = np.random.default_rng([M.SEED, 1])
pl = M.pl_sample(M.oracle_abilities(corpus, q, docs), M.TAU_NOISY, rng)
twin = M.listmle_loss_scores(M.oracle_abilities(corpus, q, docs) / M.TAU_NOISY, pl)
print("emitted NLL == imported ListMLE loss :", abs(M.emitted_nll(corpus, q, docs, pl, M.TAU_NOISY) - twin) < 1e-12)

## 2 — The sliding window is a bubble sort (RankGPT)

An LLM context fits only $w \ll n$ candidates, so RankGPT slides a window of size $w$ (step $s$)
back-to-front and locally re-sorts each window. This is a bubble/selection sort: one pass with a *perfect*
comparator bubbles the global best into the top window, and the call count is exactly
$P\cdot(\lceil (n-w)/s\rceil + 1)$ — $O(n/s)$ per pass, versus $O(n^2)$ for all-pairs. With a *noisy*
comparator the recall is not exact in one pass; it climbs with passes (monotone in expectation), and
sliding back-to-front beats front-to-back because a buried document can rise to the front in a single pass.

In [ ]:
print("call count P*(ceil((n-w)/s)+1) :", M.oracle_call_count(M.POOL, M.WINDOW, M.STEP, M.N_PASSES),
      " vs all-pairs C(n,2) :", M.POOL * (M.POOL - 1) // 2)
print("tau->0 one pass bubbles the global best into the top window :",
      all(M.perfect_pass_bubbles_best(corpus, qq) for qq in corpus["train_q"][:6]))
print("seed-averaged recall@10 by pass :",
      [(p, round(v, 3)) for p, v in M.topk_recall_vs_passes(corpus, corpus["test_q"], M.TAU_NOISY, M.N_PASSES)])
d = M.direction_asymmetry(corpus, corpus["test_q"], M.TAU_NOISY)
print("one-pass recall  back-to-front %.3f  >  front-to-back %.3f" % (d["back_to_front"], d["front_to_back"]))

## 3 — Positional bias: lost in the middle

The in-window order is biased by position — a trained transformer attends most to the ends of its context
and least to the middle (Liu et al. 2024). We model this as a per-position multiplier $b(p)$ (high at the
ends, depressed in the center) added in log-ability space. With a sharp oracle (so the bias is isolated
from sampling noise) the dip **bites** — recall drops — and presenting each window in several *random*
orders and Borda-aggregating flattens it (a center document in one presentation is at an edge in another;
note that *reversing* a window leaves its center fixed, so random presentations are required).

In [ ]:
none = M.bias_correction_recall(corpus, corpus["test_q"], "none")
biased = M.bias_correction_recall(corpus, corpus["test_q"], "biased")
corrected = M.bias_correction_recall(corpus, corpus["test_q"], "corrected")
print("recall@10  unbiased %.3f  >  biased %.3f   (the dip bites by %.3f)" % (none, biased, none - biased))
print("recall@10  corrected %.3f   (recovers toward, but does not beat, the unbiased oracle)" % corrected)
print("U-curve kernel (window center is the trough):",
      [round(v, 3) for v in M.position_bias_kernel(M.WINDOW)])

## 4 — Aggregating noisy permutations (social choice)

A reranker can emit $K$ noisy permutations (different windows, prompt orders, or samples), which must be
aggregated into one consensus. We reuse the social-choice machinery — **Borda**, **RRF**, the **Kemeny**
median (the Kendall-$\tau$-minimizing permutation, NP-hard) — and build the Dwork et al. (2001) **Markov
chain** fresh: the consensus is the *stationary distribution of a comparison random walk* (the random-walks
up-link). Aggregating $K$ ballots concentrates the consensus toward truth: the per-document averaged rank
concentrates at the $1/\sqrt{K}$ CLT rate, and the consensus Kendall-$\tau$ to $\pi^\*$ falls monotonically.

In [ ]:
for method in ("borda", "rrf", "mc4"):
    curve = M.agg_tau_vs_k(corpus, corpus["test_q"], M.candidate_set, M.TAU_NOISY, method)
    print(f"{method:6s} Kendall-tau to pi* by K :", [(K, round(t, 2)) for K, t in curve])
print("std(mean rank) * sqrt(K)  (~constant, the CLT) :",
      [(K, round(cs, 3)) for K, _, cs in M.aggregated_rank_concentration(corpus, corpus["test_q"], M.candidate_set, M.TAU_NOISY)])
print("Kemeny-cost ratios on a 6-doc set (1.0 = optimal) :", M.kemeny_approximation(corpus, corpus["train_q"][0]))

## 5 — Distilling the permutation oracle (RankVicuna / RankZephyr)

The LLM teacher is expensive ($O(n/s)$ windowed calls per query). We distill its permutations into a cheap
**linear ListMLE student** — fit the imported listwise loss to the teacher's emitted orders instead of to
$\pi^\*$. The collapse anchor: a **perfect** teacher ($\tau\to 0$) emits $\pi^\*$ for every query, so its
distilled student *is* the predecessor's own `fit_listmle` scorer, byte-for-byte. The student then answers
at **zero** inference LLM calls (the teacher's cost was paid offline).

In [ ]:
student_perfect = M.fit_student_to_teacher(corpus, corpus["train_q"], M.TAU_COLLAPSE, np.random.default_rng([M.SEED, 13]))
print("perfect-teacher student == imported fit_listmle :",
      bool(np.allclose(student_perfect, M.fit_listmle(corpus, corpus["train_q"]), atol=1e-9)))
nq = len(corpus["test_q"])
print("teacher cost %.0f   student cost %.0f   speedup %.0fx"
      % (M.teacher_call_cost(nq), M.student_call_cost(nq), M.distill_speedup()))

## 6 — The cost–quality frontier

Finally, all methods on the *same* shared pool, with cost measured in **LLM calls per query** (the model's
defining resource; the cross-encoder is the cost *unit* the LLM call is priced against, not a method here).
The honest, seed-free wins are **structural** — the $O(n/s)$ sliding-window call count versus the $O(n^2)$
all-pairs cost, and the distilled student's zero inference cost — not the aggregate scores, whose deltas sit
inside the confidence interval on this forgiving corpus.

In [ ]:
fr = M.frontier(corpus, corpus["test_q"])
for name in ("dense_best_leg", "rrf_3legs", "pointwise_llm", "sliding_window_llm", "allpairs_llm", "distilled_student"):
    m = fr[name]
    print("%-20s NDCG@10 %.3f   recall@10 %.3f   LLM calls/query %5d" % (name, m["ndcg"], m["recall"], int(m["cost"])))
v = M.verdict(fr)
print("\nsliding vs all-pairs cost ratio: %.0fx   (sliding %d calls, all-pairs %d)"
      % (v["sliding_vs_allpairs_cost_ratio"], int(v["sliding_calls"]), int(v["allpairs_calls"])))

## Where this points

This is the **terminal node** of the ranking-fusion learning-to-rank sub-track — with it the track is
complete. The thread that carries forward is one of honesty: an autoregressive permutation model, like
LambdaRank, descends *no scalar loss* at inference; the Plackett–Luce objective is what a distilled student
is trained on, not what the teacher optimizes per query. The provable content here is algorithmic — the
call-count law, the bias-correction flattening, the aggregation variance reduction, the distillation
speedup — and the notebook above re-derives each from `llm_listwise_rerankers.py`, which owns the numbers.